# Numerical checks and report for the manuscript

These tests establish geometry and controlled numerical consistency. They do not establish agreement with an experiment or a full Maxwell solver. The output JSON is the source of the quoted numerical results.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/scattering_calculator").is_dir())
PAPER = ROOT / "paper/scattering_calculator"
sys.path.insert(0, str(PAPER))
import experiments as ex
OUT = PAPER / "results"
OUT.mkdir(exist_ok=True)

def show(name):
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.imshow(plt.imread(OUT / (name + ".png")))
    ax.axis("off")
    plt.show()


## Editable experiment parameters

Change this cell and run the cells below it. Numerical values in the explanatory prose describe the original default design; the printed geometry and saved configuration are authoritative for your run. `thickness_nm=None` retunes the thickness when energy or lattice spacing changes; a number keeps it fixed. Tilt is about lab y and is measured from normal incidence. Detector centre offsets are `(x,y)` in pixels.

In [ ]:
# EDIT HERE, then Run All. These values feed every calculation below.
# Radius is where the texture reaches the uniform background, not lattice spacing.
cfg = ex.SkyrmionConfig(
    energy_eV=778.0,
    lattice_nm=12.0,
    radius_nm=3.4,
    thickness_nm=None,  # None: second-zero design; or set e.g. 100.0 / 269.48 / 400.0
    angles_deg=(-8.7947589, -4.3973795, 0.0, 4.3973795, 8.7947589),
    scan_angles_deg=tuple(np.linspace(-12, 12, 49)),  # Born + matched 3D FFT scan
    volume_angles_deg=tuple(np.linspace(-12, 12, 25)),  # production multislice scan
    detector_n=193,
    detector_pitch_m=13.5e-6,
    detector_distance_m=0.007,
    detector_center_offset_xy_px=(0.0, 0.0),
    beam_sigma_nm=24.0,  # amplitude Gaussian sigma
    born_n=256, born_dx_nm=0.75,
    multislice_n=192, multislice_dx_nm=0.75, multislice_dz_nm=2.0,
    volume_n=128,  # faster real-space grid for the multislice tilt scan
    q_bins=101, q_limit_rad_nm=0.85,
    fft_nz=512, fft_dz_nm=2.0,  # 3D FFT z box = fft_nz * fft_dz_nm; include vacuum
    contrast_channel='xmcd',  # 'mz': scalar control, no tilt-dependent projection
    include_cobalt=True,
)
# Optional: move selected image angles with the analytic Bragg condition.
# from dataclasses import replace
# tb = cfg.geometry()['bragg_deg']
# cfg = replace(cfg, angles_deg=(-2*tb, -tb, 0., tb, 2*tb))

OUT = PAPER / 'results' / 'notebook_skyrmion'  # use a different name for each experiment
OUT.mkdir(parents=True, exist_ok=True)
p = cfg.geometry()
print(p)
print('Ewald sphere misses central rod lobe:', p['outside_central_lobe'])
print('Untilting thickness factor:', p['normal_thickness_factor'])


In [ ]:
report = ex.validation(OUT, config=cfg)
print(report)

## Additional lateral refinement at fixed field of view

Compare 128×128 at 0.75 nm against 192×192 at 0.5 nm, keeping the 96 nm field of view and 2 nm longitudinal sampling fixed. The reciprocal pixel spacing is therefore identical. Compare the same +G Bragg region; do not rescale either pattern independently.


In [ ]:
# Compare identical field of view and momentum sampling.
angle = cfg.geometry()['bragg_deg']
n0 = cfg.volume_n
n1 = 2*n0
coarse = ex.skyrmion_multislice(angle, n=n0, config=cfg)
fine = ex.skyrmion_multislice(angle, n=n1, dx=cfg.multislice_dx_nm/2, config=cfg)
start = n1//2 - n0//2
fine_image = fine['intensity'][start:start+n0, start:start+n0]
g = cfg.geometry()['G_rad_nm']
q = coarse['q_lab']
region = (q[...,0]-g*np.cos(np.deg2rad(angle)))**2 + q[...,1]**2 < (.13*g)**2
error = np.linalg.norm((coarse['intensity']-fine_image)[region])/np.linalg.norm(fine_image[region])
print('Lateral refinement relative L2:', error)
import json
(OUT / 'lateral_refinement.json').write_text(json.dumps({'relative_L2':float(error)}, indent=2))


## Publication validation still needed

Check field-of-view dependence and boundary padding; converge the full rocking curve and weak-medium Born comparison quantitatively; refine FTH reference-hole sampling; add Jones tilted-volume validation and an independent solver or experimental dataset. Verify provenance and redistribution rights of the optical constants, specify authors/contributions/funding, and archive a release with a persistent identifier. These are stated as open work in the manuscript rather than presented as completed validation.
